In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Unzip from Drive straight to Colab's high-speed local storage
!unzip -q "/content/drive/MyDrive/Colab Notebooks/covid detection/archive.zip" -d "/content/dataset/"

Mounted at /content/drive


In [3]:
import os
import tensorflow as tf
import keras
from keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.models import Model
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory

In [4]:
# Define paths and constants
TRAIN_DIR = 'dataset/Data/train/'
TEST_DIR = 'dataset/Data/test/'
IMG_SIZE = (224, 224)  # VGG16 expected input size
BATCH_SIZE = 32
NUM_CLASSES = 3        # COVID19, NORMAL, PNEUMONIA

In [5]:
# Setup ImageDataGenerators with Xception preprocessing
# Data Augmentation for training to prevent overfitting
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    label_mode='categorical'  # For multi-class classification
)

# Only preprocessing (no augmentation) for testing/validation
val_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    label_mode='categorical'  # For multi-class classification
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'  # For multi-class classification
)

train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

Found 5144 files belonging to 3 classes.
Using 4116 files for training.
Found 5144 files belonging to 3 classes.
Using 1028 files for validation.
Found 1288 files belonging to 3 classes.


In [6]:
# Build Xception Transfer Learning Model
input_shape = IMG_SIZE + (3,)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the base Xception layers
base_model.trainable = False

# Build custom network head
inputs = Input(shape=input_shape)
x = preprocess_input(inputs)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,846,787 (56.64 MB)

 Trainable params: 132,099 (516.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [7]:
# Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
# Train the Model
EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

Epoch 1/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 55s 369ms/step - accuracy: 0.8460 - loss: 0.7510 - val_accuracy: 0.9553 - val_loss: 0.1387
Epoch 2/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 26s 204ms/step - accuracy: 0.9130 - loss: 0.2364 - val_accuracy: 0.9562 - val_loss: 0.1277
Epoch 3/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 27s 210ms/step - accuracy: 0.9189 - loss: 0.2060 - val_accuracy: 0.9611 - val_loss: 0.1216
Epoch 4/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 28s 216ms/step - accuracy: 0.9327 - loss: 0.1820 - val_accuracy: 0.9630 - val_loss: 0.1147
Epoch 5/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 30s 229ms/step - accuracy: 0.9483 - loss: 0.1438 - val_accuracy: 0.9660 - val_loss: 0.1068
Epoch 6/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 29s 226ms/step - accuracy: 0.9424 - loss: 0.1465 - val_accuracy: 0.9630 - val_loss: 0.1124
Epoch 7/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 29s 222ms/step - accuracy: 0.9483 - loss: 0.1398 - val_accuracy: 0.9582 - val_loss: 0.1245
Epoch 8/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 29s 228ms/step - accuracy: 0.9548 - loss: 0

In [9]:
import numpy as np
import seaborn as sns
import sklearn.metrics as metrics
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Overall Performance Evaluation
loss, accuracy = model.evaluate(test_ds)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

41/41 ━━━━━━━━━━━━━━━━━━━━ 14s 293ms/step - accuracy: 0.9356 - loss: 0.1498

Test Loss: 0.1498
Test Accuracy: 93.56%


In [10]:
# Save the trained model
model.save('covid_pneumonia_vgg16_model.keras')
print("Model saved successfully!")

Model saved successfully!
